# **SIMD与SIMT混合编程实践：Transpose算子性能优化**

## 概述

本小节介绍**SIMD与SIMT混合编程**模式下的Transpose算子开发与优化实现。

Transpose是离散访存的场景，在上一节中已经介绍了使用SIMT快速完成算子功能开发和性能优化。本节再继续围绕Transpose算子展开实践：先使用SIMD与SIMT混合编程的SIMT单元实现一个最简单的Transpose算子，然后引入MTE搬运单元，完成GM和UB间的数据搬运，并利用MTE与SIMT VF间的流水并行，进一步提升性能。

### 学习前置要求

学习本小节前，建议已经具备以下基础：

- 已学习《SIMT编程模型》中核函数、线程索引、内存层级等内容和Gather算子编程实战。
- 已学习《SIMD编程模型》中核函数、基于指针的C语言编程等内容。
- 已学习《SIMD与SIMT混合编程模型》(03.05.01-03.05.03)，理解核函数与VF函数、`asc_vf_call` 调用方式、UB内存层级。
- 了解基本的Ascend C算子开发和执行流程。

### 学习目标

完成本小节后，开发者应能够：

- 掌握混合编程的典型应用场景：由MTE搬运单元负责连续数据搬运，SIMT单元负责分支计算。
- 学会从访存连续性判断MTE的适用边界：连续访问的读/写可由MTE搬运加速，离散访问（如Scatter的写）由SIMT完成。
- 掌握基于tile划分、UB中转、UB padding、双缓冲的Transpose算子性能优化路径，并能将其迁移到其他离散访存场景。

### 本节内容

- 环境准备
- 基于混合编程的Transpose算子实现与性能优化
- 小结

## 1. 环境准备

正式开始学习之前，先执行下方脚本检查CANN Toolkit是否可用，并把CANN环境变量加载到当前Jupyter进程，保证后续能够正常导入相关代码并使用bisheng编译器完成算子的开发与编译。

本节所有编译、运行和练习修改都在 `Sources/07.06` 目录下进行，`src` 目录仅作为只读的源码仓库存放原始代码。


In [ ]:
import os
import subprocess
import shlex
from pathlib import Path


def find_cann_home():
    candidates = []
    for key in ["ASCEND_HOME_PATH", "ASCEND_TOOLKIT_HOME"]:
        value = os.environ.get(key)
        if value:
            candidates.append(Path(value).expanduser())

    candidates.extend([
        Path.home() / "Ascend/cann",
        Path.home() / "Ascend/ascend-toolkit/latest",
        Path("/usr/local/Ascend/cann"),
        Path("/usr/local/Ascend/ascend-toolkit/latest"),
    ])

    for candidate in candidates:
        normalized = candidate
        if normalized.name in {"x86_64-linux", "aarch64-linux"}:
            normalized = normalized.parent
        set_env = normalized / "set_env.sh"
        if set_env.exists():
            return normalized.resolve(), set_env.resolve()

    raise RuntimeError("未找到 CANN Toolkit，请确认已安装 CANN，并设置环境变量。")


def source_cann_env(set_env):
    command = f"set -a && source {shlex.quote(str(set_env))} >/dev/null 2>&1 && env"
    result = subprocess.run(["bash", "-lc", command], check=True, text=True, capture_output=True)
    for line in result.stdout.splitlines():
        if "=" in line:
            key, value = line.split("=", 1)
            os.environ[key] = value


cann_home, cann_set_env = find_cann_home()
source_cann_env(cann_set_env)

WORKSPACE = Path("Sources/07.06")
WORKSPACE.mkdir(parents=True, exist_ok=True)

print(f"CANN Toolkit: {cann_home}")
print(f"Workspace: {WORKSPACE.resolve()}")


## 2. Transpose算子功能介绍
Transpose算子的主要功能是实现二维数据的转置，计算公式如下：

```text
output(col, row) = input(row, col)
```

**本节实现的算子规格**：

| 项 | 取值 |
| --- | --- |
| 输入 `input` | `(1024 , 1024)`，`float` |
| 输出 `output` | `(1024 , 1024)`，`float` |


## 3. Transpose算子实现

Transpose属于典型的离散访存场景：输入按行连续读取，但转置后的输出需要跨行写入，写地址不连续。本节围绕Transpose算子，沿着一条 **4级优化路径** 展开实践：先使用SIMT直接访问GM完成最基础的实现，暴露非连续写带来的性能瓶颈；再逐步引入MTE搬运单元、UB中转、Thread Block到tile的映射调整、UB padding缓解bank冲突，最后通过双缓冲让MTE搬运与SIMT计算流水并行，逐步把Transpose算子的性能调优到接近GM带宽上限。

### 3.1 准备源码工作目录

完成环境准备并明确Transpose算子规格后，我们开始动手实践。本节按照下表列出的4个优化步骤逐步优化Transpose算子，每个步骤的实现放在独立目录中：

| 目录 | 核函数 | 优化点 |
| --- | --- | --- |
| `naive` | `transpose_naive_kernel` | SIMT直接读写GM，转置写地址跨行不连续 |
| `ub_loop` | `transpose_ub_loop_kernel` | 引入MTE搬运 + 32×32 tile + UB中转，Thread Block数固定为硬件vector core数，核内循环处理多个tile |
| `ub_pad` | `transpose_ub_pad_kernel` | 输入tile改为32×40 padding布局，降低SIMT转置读UB的bank冲突 |
| `ub_pad_db` | `transpose_ub_pad_db_kernel` | 双缓冲（ping/pong）使MTE2搬入、SIMT VF转置、MTE3搬出流水并行 |

上表4个版本的完整源码都已存放在 `src/07_06_simd_simt_transpose/` 下。执行下面的单元格，把这些源码拷贝到 `Sources/07.06/simd_simt_transpose/` 作为本节课程的工作目录，后续的编译、运行和修改都在 `Sources` 下完成：

In [ ]:
import shutil
from pathlib import Path

SRC_ROOT = Path("src/07_06_simd_simt_transpose")            # 只读源码目录
DST_ROOT = Path("Sources/07.06/simd_simt_transpose")  # 工作目录

VERSIONS = ["naive", "ub_loop", "ub_pad", "ub_pad_db"]

# 清理旧的工作目录，保证每次都从 src 拷贝出干净的一份
if DST_ROOT.exists():
    shutil.rmtree(DST_ROOT)

for version in VERSIONS:
    dst = DST_ROOT / version
    dst.mkdir(parents=True, exist_ok=True)
    for pattern in ("*.asc", "*.h", "CMakeLists.txt"):
        for f in sorted((SRC_ROOT / version).glob(pattern)):
            shutil.copy2(f, dst / f.name)
    print(f"{version}: {sorted(p.name for p in dst.iterdir())}")


### 3.2 SIMT直接访问GM实现Transpose

#### 3.2.1 实现思路与代码

首先在混合编程场景使用SIMT直接访问GM，完成最基础的Transpose实现：每个线程从 `input` 连续读取对应元素值，计算出转置后的坐标，直接写到 `output` 对应位置。

我们在07.05课程中已经得到明确的基本性能优化手段：数据量较大的场景，要限制启动的核数不超过物理核以减少额外头开销。因此本实现直接采用这一优化手段：限制启动核数在物理核以内，每个线程处理多个元素。

该实现由核函数和SIMT VF函数两部分组成，先看作为Device侧入口的核函数：

```cpp
__global__ __vector__ void transpose_naive_kernel(
    __gm__ float* output, __gm__ float* input, uint32_t width, uint32_t height)
{
    asc_init();
    uint32_t total = width * height;
    asc_vf_call<transpose_simt_naive>(dim3(THREAD_COUNT), output, input, width, height, total);
}
```

核函数接收GM上的输入输出数据地址，调用SIMT VF函数完成转置计算。

SIMT VF函数负责实际的坐标变换与读写：

```cpp
__simt_vf__ __launch_bounds__(2048) inline void transpose_simt_naive(
    __gm__ float* output, __gm__ float* input, uint32_t width, uint32_t height, uint32_t total)
{
    // for循环用于处理多个数据，也可防止越界处理
    for (uint32_t i = blockIdx.x * blockDim.x + threadIdx.x; i < total; i += gridDim.x * blockDim.x) {
        uint32_t row = i / width;
        uint32_t col = i - row * width;
        // 直接写转置后的GM地址，写入方向跨行不连续。
        output[col * height + row] = input[i];
    }
}
```

SIMT线程按grid-stride循环遍历输入矩阵的每个元素：`input[i]` 按行优先连续读取，而转置后写入的目标地址 `output[col * height + row]` 是按列跨行分布的，同一个Warp内相邻线程的写地址会落在输出矩阵的不同行，属于非连续写。由于矩阵转置本身计算量很小，这个非连续写会成为该实现的主要瓶颈。

本实现的完整代码保存在 `simd_simt_transpose_naive.asc` 中，执行下面的单元格查看完整源码：

In [ ]:
!cat Sources/07.06/simd_simt_transpose/naive/simd_simt_transpose_naive.asc

#### 3.2.2 CMake配置

接下来为当前实现编写CMake配置文件。

对应的 `CMakeLists.txt` 内容如下，同样已随源码一起拷贝到工作目录：


```cmake
cmake_minimum_required(VERSION 3.16)

set(CMAKE_ASC_ARCHITECTURES "dav-3510" CACHE STRING "NPU ARCH, e.g. dav-3510")

# find_package(ASC) 用于查找和配置 Ascend C 编译工具链
find_package(ASC REQUIRED)
# 指定项目支持的语言包括 ASC 和 CXX，ASC 表示支持使用毕昇编译器对 Ascend C 编程语言进行编译
project(transpose_naive_sample LANGUAGES ASC CXX)

add_executable(demo
    simd_simt_transpose_naive.asc
)

# 通过编译选项设置 NPU 架构
target_compile_options(demo PRIVATE
    $<$<COMPILE_LANGUAGE:ASC>:--npu-arch=${CMAKE_ASC_ARCHITECTURES}>
)
```

**编译选项说明：**

| 选项 | 说明 |
| --- | --- |
| `--npu-arch=dav-3510` | 指定NPU架构版本，`dav-` 后为架构号，Ascend 950PR/Ascend 950DT 对应 `dav-3510` |

> **注意**：在SIMD与SIMT混合编程场景中，虽然使用SIMT VF函数完成计算，但不需要添加 `--enable-simt` 编译选项。

#### 3.2.3 编译运行并采集性能

执行以下命令编译并运行当前实现：


In [ ]:
# 需在已配置 CANN 环境的 NPU 机器上执行
!cd Sources/07.06/simd_simt_transpose/naive && mkdir -p build && cd build && \
 cmake -DCMAKE_ASC_ARCHITECTURES=dav-3510 .. && make -j && \
 ./demo

编译运行成功后，若看到以下输出，则说明计算结果与预期完全一致：

```text
[Success] Case accuracy verification passed.
```

完成正确性验证后，使用 `msOpProf` 工具采集算子性能。`msopprof ./demo` 会运行可执行文件并生成 `OPPROF_{timestamp}_...` 性能数据目录，可用于查看算子基础信息、执行耗时、Pipe利用率和内存访问情况。

In [ ]:
!cd Sources/07.06/simd_simt_transpose/naive/build && msopprof ./demo

SIMT直接访问GM的实现在Ascend 950 环境、CANN 9.1.0 上实测的结果如下：

| 优化点 | 核函数 | Task Duration(μs) | aiv_vec_time(μs) | aiv_mte2_time(μs) | aiv_mte3_time(μs) |
| --- | --- | --- | --- | --- | --- |
| SIMT直接访问GM | `transpose_naive_kernel` | 36.38 | 33.17 | 0.00 | 0.00 |

该实现中，输入按行连续读取，但转置写地址跨行不连续，同一Warp内相邻线程的写地址分散到输出矩阵的不同行，导致GM写效率较低。由于矩阵转置本身计算量很小，这个非连续写会成为该实现的主要瓶颈，后续优化的核心思路是把非连续访问从GM转移到UB，使GM读写都变成连续访问。

### 3.3 引入UB中转、MTE搬运

#### 3.3.1 UB中转与MTE搬运的实现思路

上一实现的瓶颈在于转置写地址跨行不连续。本实现参照07.05中引入UB作为中转的优化思路，将GM的非连续访存转移到访问效率更高的UB上。另外，由于32*32 tile块数据相对连续规整，混合编程场景还可使用MTE进行数据搬运，这将进一步提升访存效率。

详细实现思路如下：

先将矩阵划分为32×32的tile，MTE2把一个tile从GM连续搬入UB，SIMT VF在UB内完成转置访问并写入另一块UB输出缓冲区，最后MTE3把转置后的tile按输出矩阵行方向连续搬回GM。这样一来，非连续访问被转移到了UB内部，GM侧的读和写都变成连续访问。

先看MTE搬运的辅助函数，负责把一个tile从GM搬运到UB。这里把UB侧每行的跨距 `ub_row_bytes` 做成参数，是为了让同一个函数既能搬入本节这种32×32连续布局的UB，也能在后面3.4节搬入带padding的UB（32×40布局），避免为两种布局各写一遍几乎相同的搬运逻辑：

```cpp
// 把一个32×32 tile从GM连续搬入UB，ub_row_bytes由调用方指定UB侧每行跨距，便于非padding/padding场景复用
__aicore__ inline void copy_gm_tile_to_ub(
    __ubuf__ float* in_tile, __gm__ float* input, uint32_t width, uint32_t tiles_x, uint32_t tile_id,
    uint32_t ub_row_bytes)
{
    uint32_t tile_y = tile_id / tiles_x;
    uint32_t tile_x = tile_id - tile_y * tiles_x;
    uint32_t input_offset = tile_y * TILE_DIM * width + tile_x * TILE_DIM;
    // MTE2按二维搬运把一个32×32 tile从GM连续搬入UB。
    asc_copy_gm2ub_align(
        in_tile, input + input_offset, TILE_DIM, TILE_ROW_BYTES, 0, 0, false,
        asc_load_l2_cache_mode::NORMAL_FIRST_VICTIM, width * sizeof(float), ub_row_bytes);
}
```

其中，`asc_copy_gm2ub_align` 的高维切分搬运接口按 `n_burst` 个连续数据块搬运，每块长度为 `len_burst` 字节：这里把tile的每一行当作一个数据块，`n_burst=32`，`len_burst=32*sizeof(float)`，源端相邻行之间的地址间隔 `src_stride` 为矩阵的一整行字节数，目的端UB中tile按 `ub_row_bytes` 跨距存放。本节调用时传入 `ub_row_bytes=TILE_ROW_BYTES`，即tile按32×32连续存放，因此32行数据被连续搬入UB。

`l2_cache_mode` 参数用于配置数据在L2 Cache中的管理策略，需要传入 `asc_load_l2_cache_mode` 枚举类型的值：本例使用 `NORMAL_FIRST_VICTIM`，表示启用L2 Cache并把分配的Cache Line标记为高替换优先级。把结果从UB搬回GM的 `asc_copy_ub2gm_align` 同理，对应的枚举类型为 `asc_store_l2_cache_mode`。

每个Thread Block启动的线程数为2048，`2048 / (32 * 32) = 2`, 因此整个线程块能一次处理两个tile，因此在此之上再包一层循环，依次把两个tile搬入UB：

```cpp
// 把本Thread Block分到的2个tile依次从GM搬入UB
__aicore__ inline void copy_gm_2tile_to_ub(
    __ubuf__ float* in_tile, __gm__ float* input, uint32_t width, uint32_t tiles_x, uint32_t tile_base,
    uint32_t total_tiles)
{
    for (uint32_t local_tile = 0; local_tile < TILES_PER_BLOCK; ++local_tile) {
        uint32_t tile_id = tile_base + local_tile;
        if (tile_id < total_tiles) {
            // 非padding场景下，UB中每个输入tile为连续32×32布局。
            copy_gm_tile_to_ub(in_tile + local_tile * TILE_ELEMENTS, input, width, tiles_x, tile_id, TILE_ROW_BYTES);
        }
    }
}
```


SIMT VF函数在UB内完成转置访问。由于数据搬运单独放在SIMT VF外部执行，SIMT VF内只需要完成UB上的数据转置计算，计算复杂度比SIMT场景的对应实现更简单，寄存器压力也更小，因此可以先让每个线程块启动2048个线程，共同完成一组Tile（2个Tile块），SIMT VF实现大致如下：

```cpp
// SIMT VF 函数：在UB内完成2个tile的转置访问
__simt_vf__ __launch_bounds__(THREADS_2048) inline void simt_transpose_2tile(
    __ubuf__ float* out_tile, __ubuf__ float* in_tile, uint32_t tile_base, uint32_t total_tiles)
{
    uint32_t local_tile = threadIdx.y >> 5; // 等价于 threadIdx.y / 32
    uint32_t ty = threadIdx.y & (TILE_DIM - 1); // 等价于 threadIdx.y % 32
    uint32_t tx = threadIdx.x;
    uint32_t tile_id = tile_base + local_tile;
    if (tile_id >= total_tiles) {
        return;
    }
    // SIMT VF在UB内按转置方向读取in_tile，并连续写入out_tile。
    out_tile[local_tile * TILE_ELEMENTS + ty * TILE_DIM + tx] =
        in_tile[local_tile * TILE_ELEMENTS + tx * TILE_DIM + ty];
}
```


2048个线程按 `dim3(32, 64, 1)` 组织：若`threadIdx.y` 属于[0,31]，则处理第一个tile, 若属于[32, 63]则处理第二个tile，用`threadIdx.y /32`计算出该线程处理的是哪个tile，用`threadIdx.y % 32` 计算出对应tile内的行坐标。每个线程从 `in_tile` 读取转置前的元素，写到 `out_tile` 中转置后的位置；`out_tile` 按输出tile的行方向连续排布，方便后续MTE3连续搬出。

矩阵按32×32划分后，总tile数为 `total_tiles = (width/32) × (height/32)`。本实现每个Thread Block启动2048个线程，可以同时处理两个tile，因此每个Thread Block一轮处理一组（2个）tile。

#### 3.3.2 每个核处理多个Tile的实现思路

核函数负责调度MTE搬运和SIMT VF调用。由于VF函数属于PIPE_V流水，与MTE2、MTE3流水并行，因此需要在各个流水间加上同步。

当输入矩阵较大时，为了限制启动的核数不超过物理核，需要设计处理多Tile的逻辑。

明确Thread Block的数量与tile的映射关系：

把Thread Block数限制在物理核数内（通过 `get_vector_core_num` 运行时查询），每个Thread Block在核内以 `block_num * TILES_PER_BLOCK` 为步长循环处理多组tile，直到覆盖所有tile。这样每个核只会被调度一次，不再需要额外的Thread Block排队：

```cpp
// 核函数：Thread Block数固定为物理核数，核内循环处理多组tile
__global__ __vector__ void transpose_ub_loop_kernel(
    __gm__ float* output, __gm__ float* input, uint32_t width, uint32_t height, uint32_t total_tiles)
{
    asc_init();
    __ubuf__ float in_tile[TILES_PER_BLOCK][TILE_DIM][TILE_DIM];
    __ubuf__ float out_tile[TILES_PER_BLOCK][TILE_DIM][TILE_DIM];
    uint32_t tiles_x = width / TILE_DIM;
    uint32_t loop_step = block_num * TILES_PER_BLOCK;

    // 固定 Thread Block 数为物理核数，核内循环处理多组 tile。
    for (uint32_t tile_base = block_idx * TILES_PER_BLOCK; tile_base < total_tiles; tile_base += loop_step) {
        asc_lock(PIPE_MTE2, SINGLE_BUFFER_MUTEX);
        copy_gm_2tile_to_ub(&in_tile[0][0][0], input, width, tiles_x, tile_base, total_tiles);
        asc_unlock(PIPE_MTE2, SINGLE_BUFFER_MUTEX);

        // MTE2搬入完成后，SIMT VF读取UB输入buffer。
        asc_lock(PIPE_V, SINGLE_BUFFER_MUTEX);
        asc_vf_call<simt_transpose_2tile>(
            dim3(TILE_DIM, TILE_DIM * TILES_PER_BLOCK, 1), &out_tile[0][0][0], &in_tile[0][0][0], tile_base,
            total_tiles);
        asc_unlock(PIPE_V, SINGLE_BUFFER_MUTEX);

        // SIMT VF写完输出buffer后，MTE3将结果搬回GM。
        asc_lock(PIPE_MTE3, SINGLE_BUFFER_MUTEX);
        copy_ub_2tile_to_gm(output, &out_tile[0][0][0], height, tiles_x, tile_base, total_tiles);
        asc_unlock(PIPE_MTE3, SINGLE_BUFFER_MUTEX);
    }
}
```

MTE2搬入、SIMT VF计算、MTE3搬出之间存在数据依赖，因此用同一个 `mutex_id` 通过 `asc_lock`/`asc_unlock` 约束三者的执行顺序。

完整实现代码与CMakeLists.txt已保存在 `Sources/07.06/simd_simt_transpose/ub_loop/` 目录下，此处不再重复展示全文。执行以下命令查看源码内容：

In [ ]:
!cat Sources/07.06/simd_simt_transpose/ub_loop/simd_simt_transpose_ub_loop.asc

#### 3.3.3 编译运行并采集性能

执行以下命令编译并运行当前实现：

In [ ]:
# 需在已配置 CANN 环境的 NPU 机器上执行
!cd Sources/07.06/simd_simt_transpose/ub_loop && mkdir -p build && cd build && \
 cmake -DCMAKE_ASC_ARCHITECTURES=dav-3510 .. && make -j && \
 ./demo

编译运行成功后，若看到以下输出，则说明计算结果与预期完全一致：

```text
[Success] Case accuracy verification passed.
```

完成正确性验证后，使用 `msOpProf` 工具采集算子性能。

In [ ]:
!cd Sources/07.06/simd_simt_transpose/ub_loop/build && msopprof ./demo

本实现在Ascend 950 环境、CANN 9.1.0 上实测的结果如下：

| 优化点 | 核函数 | Task Duration(μs) | aiv_vec_time(μs) | aiv_mte2_time(μs) | aiv_mte3_time(μs) |
| --- | --- | --- | --- | --- | --- |
| SIMT直接访问GM | `transpose_naive_kernel` | 36.38 | 33.17 | 0.00 | 0.00 |
| 引入UB中转、MTE搬运 | `transpose_ub_loop_kernel` | 18.77 | 11.56 | 3.39 | 2.05 |

引入MTE搬运和UB中转后，GM侧的读写都变成连续访问，转置计算被限制在UB内部完成，Task Duration相比SIMT直接访问GM的实现方式大幅下降（36.38μs → 18.77μs），说明UB内的转置访问远比直接读写GM高效。

不过，`simt_transpose_2tile` 中SIMT VF转置读取 `in_tile` 时，同一时刻32个线程按列访问一个32×32的UB tile，这些访问地址在UB的bank/subbank结构上会集中落在少数bank上，产生读读bank冲突，从而限制了 `aiv_vec_time` 的下降空间。下一步将通过在UB中为输入tile增加padding，改变数据的物理布局来消除这种bank冲突。

### 3.4 UB padding缓解bank冲突

#### 3.4.1 UB bank结构与bank冲突原理

上一步已经把Thread Block数固定为物理核数，消除了调度层面的额外开销，但 `simt_transpose_2tile` 内部的转置访问仍然存在瓶颈：SIMT VF转置读取 `in_tile` 时，同一时刻32个线程按同一列、不同行读取一个32×32的tile，这些访问会集中落到少数相同的bank/subbank资源上，产生读读冲突，限制了并行读取的带宽。

以Ascend 950PR/Ascend 950DT 为例，UB划分为16个bank，并组织为8个bank group，每个bank又划分为4个subbank。SIMT VF内若同一个Warp内多个线程在同一条UB访问指令中访问同一个bank group的相同编号subbank，硬件需要排队处理，从而形成subbank冲突并增加访问延迟。

<img src="./images/07_06_simd_simt_transpose/bank_structure.png" alt="bank_structure"  width="1500px" >

SIMT VF内访问UB时的bank冲突为更细粒度的subbank冲突，主要有以下两类：

- **写写冲突**：多个写操作同时访问同一个bank group的相同编号subbank。
- **读读冲突**：多个读操作同时访问同一个bank group的相同编号subbank。

以未做padding的32×32 tile为例，`in_tile` 在UB中按行优先存储，每行32个 `float` 共 `TILE_DIM * sizeof(float) = 128` 字节，恰好跨越4个bank。按照UB的地址低位交织规则，`in_tile` 的第一行覆盖bank0～bank3，第二行覆盖bank4～bank7，第三行覆盖bank8～bank11，其余行依次类推。SIMT VF转置时，同一个Warp的32个线程读取 `in_tile` 的一列元素，由于行跨度固定为32个 `float`，这32次访问会集中落到两个bank group的subbank0上，属于读读冲突场景，如下图所示：

<img src="./images/07_06_simd_simt_transpose/case2_bank.png" alt="case2_bank"  width="1500px" >

要缓解这种冲突，需要改变 `in_tile` 的物理行跨度，让同一列的相邻元素在UB物理地址上错开分布。SIMT编程 场景下，对32×32的 `float` tile只需增加一个subbank宽度、即2列padding（32×34布局）即可避免subbank冲突；但在SIMD与SIMT混合编程场景下，`in_tile` 同时是MTE2搬运的目的端，MTE搬运带padding的二维数组时还需要考虑UB中相邻行的地址跨度对齐。32×34布局的行跨度为 `34 * sizeof(float) = 136` 字节，不满足32字节对齐；因此本实现把padding列数设为8，采用32×40布局，行跨度为 `40 * sizeof(float) = 160` 字节、对应20个subbank，既能错开UB bank访问，也满足MTE搬运对UB行跨度的对齐要求。转置访问同一列时，相邻行元素按20个subbank的跨度错开，32个线程的访问不再集中到相同bank group的同一编号subbank，从而降低SIMT VF转置访问阶段的subbank冲突：

<img src="./images/07_06_simd_simt_transpose/case3_bank.png" alt="case3_bank"  width="1500px" >

需要注意的是，32×40布局是SIMD与SIMT混合编程场景下结合MTE搬运对齐要求后的折中选择。与SIMT编程 场景增加2列padding形成32×34布局不同，32×40布局仍可能存在少量subbank冲突，但冲突强度已经明显低于未padding的32×32布局。

#### 3.4.2 实现思路与代码

本实现沿用上一步固定Thread Block数、核内循环处理多组tile的结构，仅将输入tile的UB布局从32×32改为32×40（`TILE_PAD_STRIDE = TILE_DIM + TILE_PAD`，`TILE_PAD = 8`）。输出tile `out_tile` 仍然按32×32连续布局存放，因为SIMT VF按输出tile的行方向连续写入，可通过访存合并降低写入开销，不需要增加padding。

搬入辅助函数复用3.3节的 `copy_gm_tile_to_ub`（带 `ub_row_bytes` 参数），只是调用时把 `ub_row_bytes` 改传为 `TILE_PAD_STRIDE * sizeof(float)`，让MTE2按padded布局写入UB：


```cpp
__aicore__ inline void copy_gm_2tile_to_padded_ub(
    __ubuf__ float* in_tile, __gm__ float* input, uint32_t width, uint32_t tiles_x, uint32_t tile_base,
    uint32_t total_tiles)
{
    for (uint32_t local_tile = 0; local_tile < TILES_PER_BLOCK; ++local_tile) {
        uint32_t tile_id = tile_base + local_tile;
        if (tile_id < total_tiles) {
            // padding场景下，输入tile使用32×40布局以降低转置读UB时的bank冲突。
            copy_gm_tile_to_ub(
                in_tile + local_tile * TILE_PAD_ELEMENTS, input, width, tiles_x, tile_id,
                TILE_PAD_STRIDE * sizeof(float));
        }
    }
}
```


与上一步的调用方式相比，唯一的差异是传给 `copy_gm_tile_to_ub` 的最后一个参数 `ub_row_bytes` 从 `TILE_ROW_BYTES`（32×4=128字节）改为 `TILE_PAD_STRIDE * sizeof(float)`（40×4=160字节），即每搬入一行后，UB中下一行的起始地址跳过padding部分；`copy_gm_tile_to_ub` 本身的实现完全不用改动，这也是把 `ub_row_bytes` 做成参数的意义所在。

SIMT VF转置函数按padded布局读取 `in_tile`，写入仍是连续的32×32 `out_tile`：


```cpp

__simt_vf__ __launch_bounds__(THREADS_2048) inline void simt_transpose_2tile_pad(
    __ubuf__ float* out_tile, __ubuf__ float* in_tile, uint32_t tile_base, uint32_t total_tiles)
{
    uint32_t local_tile = threadIdx.y >> 5;
    uint32_t ty = threadIdx.y & (TILE_DIM - 1);
    uint32_t tx = threadIdx.x;
    uint32_t tile_id = tile_base + local_tile;
    if (tile_id >= total_tiles) {
        return;
    }
    // 输入tile带padding，按32×40物理布局读取；输出tile仍按32×32连续布局写入。
    out_tile[local_tile * TILE_ELEMENTS + ty * TILE_DIM + tx] =
        in_tile[local_tile * TILE_PAD_ELEMENTS + tx * TILE_PAD_STRIDE + ty];
}
```


核函数结构与上一步完全一致，只是把UB数组声明从 `[TILE_DIM][TILE_DIM]` 改为 `[TILE_DIM][TILE_PAD_STRIDE]`，并调用padded版本的搬运函数和SIMT VF：

```cpp
__global__ __vector__ void transpose_ub_pad_kernel(
    __gm__ float* output, __gm__ float* input, uint32_t width, uint32_t height, uint32_t total_tiles)
{
    asc_init();
    __ubuf__ float in_tile[TILES_PER_BLOCK][TILE_DIM][TILE_PAD_STRIDE];
    __ubuf__ float out_tile[TILES_PER_BLOCK][TILE_DIM][TILE_DIM];
    uint32_t tiles_x = width / TILE_DIM;
    uint32_t loop_step = block_num * TILES_PER_BLOCK;

    for (uint32_t tile_base = block_idx * TILES_PER_BLOCK; tile_base < total_tiles; tile_base += loop_step) {
        asc_lock(PIPE_MTE2, SINGLE_BUFFER_MUTEX);
        copy_gm_2tile_to_padded_ub(&in_tile[0][0][0], input, width, tiles_x, tile_base, total_tiles);
        asc_unlock(PIPE_MTE2, SINGLE_BUFFER_MUTEX);

        asc_lock(PIPE_V, SINGLE_BUFFER_MUTEX);
        asc_vf_call<simt_transpose_2tile_pad>(
            dim3(TILE_DIM, TILE_DIM * TILES_PER_BLOCK, 1), &out_tile[0][0][0], &in_tile[0][0][0], tile_base,
            total_tiles);
        asc_unlock(PIPE_V, SINGLE_BUFFER_MUTEX);

        asc_lock(PIPE_MTE3, SINGLE_BUFFER_MUTEX);
        copy_ub_2tile_to_gm(output, &out_tile[0][0][0], height, tiles_x, tile_base, total_tiles);
        asc_unlock(PIPE_MTE3, SINGLE_BUFFER_MUTEX);
    }
}
```

完整实现代码与CMakeLists.txt已保存在 `Sources/07.06/simd_simt_transpose/ub_pad/` 目录下，此处不再重复展示全文。

In [ ]:
!cat Sources/07.06/simd_simt_transpose/ub_pad/simd_simt_transpose_ub_pad.asc

#### 3.4.3 编译运行并采集性能

完成CMake配置后，执行以下命令编译并运行当前实现：

In [ ]:
# 需在已配置 CANN 环境的 NPU 机器上执行
!cd Sources/07.06/simd_simt_transpose/ub_pad && mkdir -p build && cd build && \
 cmake -DCMAKE_ASC_ARCHITECTURES=dav-3510 .. && make -j && \
 ./demo

编译运行成功后，若看到以下输出，则说明计算结果与预期完全一致：

```text
[Success] Case accuracy verification passed.
```

完成正确性验证后，使用 `msOpProf` 工具采集算子性能。

In [ ]:
!cd Sources/07.06/simd_simt_transpose/ub_pad/build && msopprof ./demo

UB padding缓解bank冲突的实现在Ascend 950 环境、CANN 9.1.0 上实测的结果如下（单次采集结果会受设备状态影响，实际数值以本机采集为准）：

| 优化点 | 核函数 | Task Duration(μs) | aiv_vec_time(μs) | aiv_mte2_time(μs) | aiv_mte3_time(μs) |
| --- | --- | --- | --- | --- | --- |
| SIMT直接访问GM | `transpose_naive_kernel` | 36.38 | 33.17 | 0.00 | 0.00 |
| 引入UB中转、MTE搬运 | `transpose_ub_loop_kernel` | 18.77 | 11.56 | 3.39 | 2.05 |
| UB padding缓解bank冲突 | `transpose_ub_pad_kernel` | 11.55 | 4.47 | 3.33 | 2.03 |

将输入tile的UB布局从32×32改为32×40后，SIMT VF转置读取时不再集中命中两个bank group的subbank0，读读冲突强度明显降低，`aiv_vec_time` 相比上一步明显下降（11.56μs → 4.47μs），`aiv_mte2_time`/`aiv_mte3_time` 基本不变（搬运数据量不变，只是目的端跨步略有增加），Task Duration相比上一步进一步下降（18.77μs → 11.55μs）。

不过，本实现中MTE2搬入、SIMT VF转置、MTE3搬出这三个步骤仍然共用同一组 `in_tile`/`out_tile` 缓冲区，通过同一个 `mutex_id` 串行执行：当前tile的MTE3搬出还没完成，下一轮循环的MTE2就无法开始搬入，三个pipe之间没有重叠，整体耗时约等于三者耗时之和（4.47 + 3.33 + 2.03 ≈ 9.83μs，加上循环调度等开销后接近实测的11.55μs）。

本实现的仿真指令流水图如下图所示：

<img src="./images/07_06_simd_simt_transpose/case3_trace.png" alt="case3_trace"  width="1500px" >

图中可以看到，`transpose_ub_pad_kernel` 使用单组缓冲区串行处理每组tile：MTE2搬入完成后，SIMT VF才能开始读取输入buffer；SIMT VF计算完成后，MTE3才能搬出输出buffer；MTE3搬出结束后，才能复用同一组UB buffer进入下一轮MTE2搬入。因此MTE2、SIMT VF、MTE3三条流水之间存在明显的串行等待，SIMT VF的执行间隔中会出现较多由搬运和同步带来的空隙。下一步将引入ping/pong双缓冲，让MTE2搬入下一组tile与当前组的SIMT VF转置、MTE3搬出并行执行，进一步压缩整体耗时。

### 3.5 双缓冲实现流水并行

#### 3.5.1 实现思路与代码

上一步（UB padding缓解bank冲突的实现）中MTE2搬入、SIMT VF转置、MTE3搬出三个步骤共用同一组UB缓冲区，通过同一个 `mutex_id` 强制串行：必须等当前tile组的MTE3搬出完成，才能开始下一组的MTE2搬入。本实现引入 **ping/pong双缓冲**：准备两组独立的 `in_tile`/`out_tile` 缓冲区，让当前循环的MTE2搬入使用一组缓冲区时，上一轮的SIMT VF转置和MTE3搬出可以使用另一组缓冲区并行执行，从而让 `PIPE_MTE2`、`PIPE_V`、`PIPE_MTE3` 三条流水线重叠起来。

双缓冲需要两组独立的mutex：输入缓冲区的锁和输出缓冲区的锁分别管理，且每组缓冲区各自有一把锁。核函数结构如下：

```cpp
__global__ __vector__ void transpose_ub_pad_db_kernel(
    __gm__ float* output, __gm__ float* input, uint32_t width, uint32_t height, uint32_t total_tiles)
{
    asc_init();
    // 两组UB buffer轮换使用：当前buffer进入SIMT VF/MTE3流水，下一组buffer由MTE2搬入。
    __ubuf__ float in_tile[2][TILES_PER_BLOCK][TILE_DIM][TILE_PAD_STRIDE];
    __ubuf__ float out_tile[2][TILES_PER_BLOCK][TILE_DIM][TILE_DIM];
    uint32_t tiles_x = width / TILE_DIM;
    uint32_t loop_step = block_num * TILES_PER_BLOCK;
    uint32_t loop_count = 0;

    for (uint32_t tile_base = block_idx * TILES_PER_BLOCK; tile_base < total_tiles; tile_base += loop_step) {
        uint32_t curr_buffer = loop_count & 1;
        uint8_t input_mutex = DB_INPUT_MUTEX_BASE + static_cast<uint8_t>(curr_buffer);
        uint8_t output_mutex = DB_OUTPUT_MUTEX_BASE + static_cast<uint8_t>(curr_buffer);

        asc_lock(PIPE_MTE2, input_mutex);
        copy_gm_2tile_to_padded_ub(&in_tile[curr_buffer][0][0][0], input, width, tiles_x, tile_base, total_tiles);
        asc_unlock(PIPE_MTE2, input_mutex);

        // SIMT VF等待当前输入buffer搬入完成；复用输出buffer前等待其上一次MTE3搬出完成。
        asc_lock(PIPE_V, input_mutex);
        asc_lock(PIPE_V, output_mutex);
        asc_vf_call<simt_transpose_2tile_pad>(
            dim3(TILE_DIM, TILE_DIM * TILES_PER_BLOCK, 1), &out_tile[curr_buffer][0][0][0],
            &in_tile[curr_buffer][0][0][0], tile_base, total_tiles);
        asc_unlock(PIPE_V, input_mutex);
        asc_unlock(PIPE_V, output_mutex);

        asc_lock(PIPE_MTE3, output_mutex);
        copy_ub_2tile_to_gm(output, &out_tile[curr_buffer][0][0][0], height, tiles_x, tile_base, total_tiles);
        asc_unlock(PIPE_MTE3, output_mutex);
        ++loop_count;
    }
}
```


与上一步相比，主要变化是：

- `in_tile`/`out_tile` 各声明2组（下标0、1），`curr_buffer = loop_count & 1` 按循环次数在两组之间轮换。
- 输入缓冲区和输出缓冲区各用一对独立的 `mutex_id`（`DB_INPUT_MUTEX_BASE`/`DB_OUTPUT_MUTEX_BASE` 加上 `curr_buffer` 偏移），而不是像上一步那样所有pipe共用一把锁。这样当前轮的MTE2只需要等待"同一组"输入缓冲区上一次的读取完成，不需要等待另一组缓冲区的状态，从而让不同组的MTE2/MTE3可以与另一组的SIMT VF计算重叠执行。
- SIMT VF转置需要同时持有输入缓冲区和输出缓冲区的锁：读取当前组 `in_tile` 前要等待对应的MTE2搬入完成，写入当前组 `out_tile` 前要等待该组上一轮的MTE3搬出完成（避免覆盖尚未搬出的数据）。

MTE搬运辅助函数（`copy_gm_2tile_to_padded_ub`、`copy_ub_2tile_to_gm`）和SIMT VF函数（`simt_transpose_2tile_pad`）都直接复用上一步的实现，无需改动。

完整实现代码与CMakeLists.txt已保存在 `Sources/07.06/simd_simt_transpose/ub_pad_db/` 目录下，此处不再重复展示全文。

In [ ]:
!cat Sources/07.06/simd_simt_transpose/ub_pad_db/simd_simt_transpose_ub_pad_db.asc

#### 3.5.2 编译运行并采集性能

完成CMake配置后，执行以下命令编译并运行当前实现：

In [ ]:
# 需在已配置 CANN 环境的 NPU 机器上执行
!cd Sources/07.06/simd_simt_transpose/ub_pad_db && mkdir -p build && cd build && \
 cmake -DCMAKE_ASC_ARCHITECTURES=dav-3510 .. && make -j && \
 ./demo

编译运行成功后，若看到以下输出，则说明计算结果与预期完全一致：

```text
[Success] Case accuracy verification passed.
```

完成正确性验证后，使用 `msOpProf` 工具采集算子性能。

In [ ]:
!cd Sources/07.06/simd_simt_transpose/ub_pad_db/build && msopprof ./demo

双缓冲流水并行的实现在Ascend 950 环境、CANN 9.1.0 上实测的结果如下（单次采集结果会受设备状态影响，实际数值以本机采集为准）：

| 优化点 | 核函数 | Task Duration(μs) | aiv_vec_time(μs) | aiv_mte2_time(μs) | aiv_mte3_time(μs) |
| --- | --- | --- | --- | --- | --- |
| SIMT直接访问GM | `transpose_naive_kernel` | 36.38 | 33.17 | 0.00 | 0.00 |
| 引入UB中转、MTE搬运 | `transpose_ub_loop_kernel` | 18.77 | 11.56 | 3.39 | 2.05 |
| UB padding缓解bank冲突 | `transpose_ub_pad_kernel` | 11.55 | 4.47 | 3.33 | 2.03 |
| 双缓冲流水并行 | `transpose_ub_pad_db_kernel` | 6.87 | 4.33 | 3.80 | 2.52 |

引入ping/pong双缓冲后，`PIPE_MTE2`、`PIPE_V`、`PIPE_MTE3` 三条流水线可以相互重叠：当前组的SIMT VF转置和MTE3搬出执行时，下一组的MTE2搬入已经在另一组缓冲区上开始。Task Duration相比上一步进一步下降（11.55μs → 6.87μs），且这个值远低于三条pipe耗时之和（4.33 + 3.80 + 2.52 ≈ 10.65μs），说明三条流水线确实发生了重叠；但6.87μs仍高于三者中最长的 `aiv_vec_time`（4.33μs），说明重叠不是完全理想的（tile之间的锁同步、循环调度等仍会带来一部分无法隐藏的开销），实测收益介于"完全串行（约10.65μs）"和"理想全重叠（约4.33μs）"之间。

本实现的仿真指令流水图如下图所示：

<img src="./images/07_06_simd_simt_transpose/case4_trace.png" alt="case4_trace"  width="1500px" >

对比上一步（单缓冲）与本步（双缓冲）的两张流水图可以看到：单缓冲版本中MTE2、SIMT VF、MTE3依次首尾相接，下一轮MTE2必须等本轮MTE3完全结束才能开始；双缓冲版本中，当前组的SIMT VF转置、MTE3搬出正在执行时，下一组的MTE2搬入已经提前在另一组缓冲区上发起，MTE2、SIMT VF、MTE3在相邻轮次之间形成了重叠。图中仍然保留少量等待间隙，这部分等待用于保护buffer复用（同一组缓冲区被再次写入前必须等待对应的读取/搬出完成）以及跨流水的数据依赖，对应前面提到的6.87μs与理想全重叠4.33μs之间的差距。

至此，Transpose算子从最初的SIMT直接访问GM（36.38μs），经过MTE + UB中转、UB padding缓解bank冲突，最终通过双缓冲实现了流水并行（6.87μs），Task Duration下降到初始版本的约19%，完整走完了本节设计的4级优化路径。

**与理论带宽上限的对比**：本实现每次搬运读、写各一遍 `1024×1024` 的 `float` 矩阵，总数据量 `D = 1024 × 1024 × 4B × 2 = 8.39MB`。按Ascend 950PR 理论GM峰值带宽1.6TB/s计算，理论下限耗时为：

```text
T_theory = 8.39MB / 1.6TB/s ≈ 5.243μs
```

本实现最终Task Duration为6.87μs，对应等效GM读写带宽约为：

```text
8.39MB / 6.87μs ≈ 1.22TB/s
```

约达到理论峰值带宽的 `1.22 / 1.6 ≈ 76%`。这个结果已经比较接近理论带宽上限，但仍高于5.243μs的理论耗时下限，差距主要来自：UB内部的读写开销、`PIPE_MTE2`/`PIPE_V`/`PIPE_MTE3` 之间的同步等待、`asc_vf_call` 调用本身的开销、tile坐标的地址计算，以及双缓冲下流水并未完全重叠（如前面流水图中仍存在的等待间隙）。这些开销大多是当前实现路径下比较固定的成本，如果要进一步逼近理论带宽，需要从减少同步次数、增大单次搬运粒度等方向继续探索，但收益会越来越有限。

## 4. 小结

本节以Transpose算子为例，展示了SIMD与SIMT混合编程中一条典型的性能优化路径：

- **SIMT直接访问GM**：实现最简单直观，但转置写地址跨行不连续，GM写效率低，暴露了离散访存场景下的性能瓶颈。
- **MTE搬运 + UB中转**：引入MTE搬运单元和32×32 tile划分，把转置计算限制在UB内部完成：GM侧的读和写都变成连续访问，SIMT只负责UB内的离散重排。
- **UB padding缓解bank冲突**：针对SIMT转置读取UB时的bank/subbank冲突，把输入tile的物理布局由32×32改为同时满足MTE对齐要求的32×40 padding，进一步释放了SIMT计算侧的并行读取带宽。
- **双缓冲流水并行**：引入ping/pong双缓冲，让MTE2搬入、SIMT VF转置、MTE3搬出三条流水线相互重叠，把原本串行的三段耗时压缩到接近其中最长的一段。

四个优化步骤层层递进，共同印证了SIMD与SIMT混合编程的适用的典型场景：**MTE负责连续数据搬运，SIMT负责局部的离散计算与访存**。Transpose算子的离散性来自转置这一固定的几何映射，因此可以通过tile化和UB中转，把原本发生在GM上的离散写"转移"到UB内部，再借助MTE把GM侧的读写都还原为连续访问；物理存储布局、流水并行度则是在此基础上进一步压榨性能。

本节的优化路径是在 `1024 x 1024` 这一个固定shape上展开的。而双缓冲这类依赖"多轮迭代重叠"的手段，其收益和每个Thread Block的循环轮数直接相关，因此会随shape缩小而缩水。下面的课后练习让你在一个更小的 `512 x 512` shape上重新走一遍这条路径，亲手验证这一点。

如果你想在一个访存模式完全不同的算子上综合运用07.05节和本节的全部技巧，可以继续学习07.07节的大课程作业——那里以MaxPool算子为题，要求在SIMT编程 和混合编程两种模式下独立完成完整的优化链路，并会得到一些和Transpose截然不同的结论（包括一次"引入UB中转反而变慢"的负优化）。

## 课后编程习题：极小shape下的混合编程优化

### 习题背景

课程中的优化路径都是在 `1024 x 1024` 输入上验证的，每个Thread Block处理 `TILES_PER_BLOCK = 2` 个tile，循环 `(1024/32)^2 / (64*2) = 8` 轮。本习题把shape缩小到 **`128 x 256` 这个非方阵小矩阵**，请你在这个极端小shape上重新实现UB padding + 双缓冲版本，观察会发生什么。

**为什么选择这个shape？** 关键在于需要思考小shape场景如何平衡并行资源来达到性能最优：`128 x 256` 共产生 **32个tile**，很明显，tile块数远小于物理核数，需要考虑每个线程块处理几个tile块更合适？
- 方案1：沿用课程中的方案，一个线程块处理2个tile块，那么启动的AIV只有16个；
- 方案2：每个线程块处理1个tile，那么启动的AIV是32个。
方案1是启动更少的核数但每核处理的任务量较大，方案2是启动更多的核去并行分摊任务但会带来额外的头开销。

| shape | tile总数 | 每Block处理tile数 | Thread Block数 | 每Block循环轮数 |
| --- | --- | --- | --- | --- |
| `1024 x 1024` | 1024 | 2 | 64 | `1024 / (64*2) = 8` |
| **`128 x 256`** | **32** | **1** | **32** | **`32 / (32*1) = 1`** |
| **`128 x 256`** | **32** | **2** | **16** | **`32 / (16*2) = 1`** |

### 规格

| 项 | 取值 |
| --- | --- |
| 核函数名 | `transpose_ub_pad_db_kernel` |
| 输入 `input` | **(128, 256)**，`float` |
| 输出 `output` | **(256, 128)**，`float` |
| tile大小 | `32 x 32`，输入tile UB布局 `32 x 40`（MTE 32B对齐padding） |
| 每Block处理tile数 | `TILES_PER_BLOCK = 1` |
| Thread Block数 | **32**（= `32 tiles / 1`，与SIMT版本核数利用相同） |

### 要求

请参照本节3.5节 `transpose_ub_pad_db_kernel` 的双缓冲结构，在 `128 x 256` shape上补全骨架中核函数循环体内的三处 `TODO`（MTE2搬入、SIMT VF转置、MTE3搬出，含各自的 `asc_lock`/`asc_unlock`）。三个搬运/计算辅助函数都已实现完整，你只需要在核函数里正确地加锁、调用、解锁。

完成方案1的实现后，再**动手把它改写成方案2**，用 `msopprof` 采集两者的Task Duration做对比，判断在这个小shape下哪种并行资源分配更优。

改写方案2时注意：把 `TILES_PER_BLOCK` 改成1只是第一步，代码里针对"多tile"的那套结构都要跟着简化，否则会残留无谓的开销——MTE搬运函数里只迭代一次的 `for (local_tile...)` 循环、恒为0的 `local_tile * TILE_PAD_ELEMENTS` 偏移、SIMT VF里恒等于0的 `local_tile = threadIdx.y >> 5`，以及与实际线程数不符的 `__launch_bounds__(2048)`。具体改动清单见下面方案2的步骤说明。

骨架代码本身可以编译通过（但结果不正确），可以先跑通编译流程。


### 准备工作目录

执行下面的单元格，把骨架代码拷贝到工作目录：


In [ ]:
import shutil
from pathlib import Path

SRC = Path("src/07_06_transpose_128x256_simd_simt")
DST = Path("Sources/07.06/transpose_128x256")

if DST.exists():
    shutil.rmtree(DST)
DST.mkdir(parents=True, exist_ok=True)
for pattern in ("*.asc", "CMakeLists.txt"):
    for f in sorted(SRC.glob(pattern)):
        shutil.copy2(f, DST / f.name)
print(sorted(p.name for p in DST.iterdir()))


### 方案1：`TILES_PER_BLOCK = 2`，启动16个核

骨架默认就是方案1的结构（每个Thread Block处理2个tile）。查看骨架，补全核函数循环体内的三处 `TODO`：


In [ ]:
!cat Sources/07.06/transpose_128x256/transpose_128x256_simd_simt.asc

补全后编译运行，先确认功能正确：

In [ ]:
!cd Sources/07.06/transpose_128x256 && mkdir -p build && cd build && \
 cmake -DCMAKE_ASC_ARCHITECTURES=dav-3510 .. && make -j && \
 ./demo


功能正确时会看到 `[Success] Case accuracy verification passed.`。

用 `msopprof` 采集方案1的性能数据：


In [ ]:
!cd Sources/07.06/transpose_128x256/build && msopprof ./demo

在输出的 `Operator Basic Information` 中记录两项关键数据：

- `Task Duration(us)`：核函数实际执行耗时
- `Block Dim`：实际启动的核数，方案1应为 **16**

### 方案2：`TILES_PER_BLOCK = 1`，启动32个核

现在把方案1改写成方案2。请自己动手修改 `Sources/07.06/transpose_128x256/transpose_128x256_simd_simt.asc`，需要改动的地方包括：

1. `TILES_PER_BLOCK` 改为 `1`（`num_blocks = total_tiles / TILES_PER_BLOCK` 会自动变成32）。
2. **MTE2搬入函数**：只搬1个tile，去掉 `for (local_tile...)` 循环与 `local_tile * TILE_PAD_ELEMENTS` 偏移，直接用 `tile_id`。
3. **MTE3搬出函数**：同样去掉循环与 `local_tile * TILE_ELEMENTS` 偏移。
4. **SIMT VF函数**：`__launch_bounds__` 从 `2048` 改为 `1024`；`threadIdx.y` 直接就是tile内行号，删掉 `local_tile = threadIdx.y >> 5` 和 `threadIdx.y & (TILE_DIM - 1)`，索引里所有 `local_tile * ...` 的偏移一并移除。
5. **核函数**：UB数组去掉 `TILES_PER_BLOCK` 这一维（`in_tile[2][TILE_DIM][TILE_PAD_STRIDE]`）；循环变量从 `tile_base` 改为 `tile_id`，步长改为 `block_num`；`asc_vf_call` 的线程组织改为 `dim3(TILE_DIM, TILE_DIM, 1)`。

改完后重新编译运行并采集性能数据：


In [ ]:
!cd Sources/07.06/transpose_128x256/build && make -j && ./demo

In [ ]:
!cd Sources/07.06/transpose_128x256/build && msopprof ./demo

这次 `Block Dim` 应为 **32**。

### 实测结果对比

把两次采集的数据填入下表：

| 方案 | `TILES_PER_BLOCK` | Block Dim（实际核数） | 每核处理tile数 | Task Duration(μs) |
| --- | --- | --- | --- | --- |
| 方案1 | 2 | 16 | 2 | |
| 方案2 | 1 | 32 | 1 | |

完成实测后，请尝试回答下面的思考题，然后查看后面的参考数据与分析。


### 实测参考数据与分析

本习题在Ascend 950 环境、CANN 9.2.0 上的实测参考值（`msopprof` 采集，各测5次取均值）：

| 方案 | Block Dim | Task Duration(μs) | 相对方案1 |
| --- | --- | --- | --- |
| 方案1（16核 × 2 tile） | 16 | **2.358** | 基准 |
| 方案2（32核 × 1 tile，针对单tile特化） | 32 | **2.194** | **快7.0%** |

**方案2优于方案1（快7.0%）。** 在这个小shape下，`128 x 256` 只有32个tile，方案1只能启动16个核，一半的AIV处于闲置状态；方案2启动32个核，把并行资源利用起来了。虽然Thread Block数翻倍会带来额外的头尾调度开销，但在核数严重不足的情况下，**提升并行度的收益明显更大**。

需要强调的是，方案2的收益依赖于代码结构与"每核1个tile"这个新策略相匹配。如果只把 `TILES_PER_BLOCK` 改成1而保留原来的多tile结构，就会残留三类无谓开销：只迭代一次的 `for (local_tile...)` 循环和恒为0的偏移计算、SIMT VF里恒等于0的 `local_tile = threadIdx.y >> 5`、以及与实际线程数不符的 `__launch_bounds__(2048)`（会影响编译器的寄存器分配决策）。**调整并行策略时不能只改配置常量，还要让代码结构跟着一起改。**

另外注意：**两种方案下双缓冲都完全失效**。方案1的循环轮数是 `32/(16*2) = 1`，方案2是 `32/(32*1) = 1`，都只有一轮迭代。双缓冲的作用是让"当前迭代的MTE3搬出"与"下一迭代的MTE2搬入"重叠，但只有一轮时根本没有下一迭代，ping/pong的两组缓冲区只用到了其中一组。你可以把双缓冲改回单缓冲验证：耗时不会有可辨识的变化。

### 思考题

1. 方案1只启动了16个核，而硬件有64个物理核。为什么不能通过"多启动一些核"来改善？（提示：想一想tile总数是多少，一个tile能否被两个核同时处理）

2. 方案2相对方案1有"并行度翻倍"的收益，也有"Thread Block数翻倍导致调度开销上升"和"单核搬运量从8KB降到4KB使MTE固定开销占比上升"的代价。实测结果是净收益7.0%。如果shape再小一些（例如 `64 x 128`，只有8个tile），你认为这个净收益会变大还是变小？

4. 如果把shape扩大到 `256 x 512`，tile总数变为 `(256/32) * (512/32) = 8 * 16 = 128`：
   - 方案1（每核2 tile）需要多少个Thread Block？每核循环几轮？
   - 方案2（每核1 tile）需要多少个Thread Block？这超过物理核数64了吗？会发生什么？
   - 在这个shape下你预期哪个方案更优？双缓冲会开始有收益吗？
   可以动手改shape验证你的推断。

5. 从本习题可以总结出一条小shape场景的调优原则：当tile总数远小于物理核数时，应该优先保证什么？当tile总数远大于物理核数时，优化的重点又该转向哪里？

**参考答案：**

两种方案的完整实现分别保存在 `answer/07_06_transpose_128x256_solution1/`（方案1）和 `answer/07_06_transpose_128x256_solution2/`（方案2，单tile特化版本）。建议对照两份代码，重点看方案2是如何针对"每核1个tile"简化搬运函数、SIMT VF和核函数结构的。


In [ ]:
# 方案1：TILES_PER_BLOCK = 2，启动 16 个核（通用的多 tile 结构）
!cat answer/07_06_transpose_128x256_solution1/transpose_128x256_simd_simt_sol1.asc

In [ ]:
# 方案2：TILES_PER_BLOCK = 1，启动 32 个核（针对单 tile 特化）
!cat answer/07_06_transpose_128x256_solution2/transpose_128x256_simd_simt_sol2.asc